In [14]:
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
import time
import threading
import pandas as pd
import os
from datetime import datetime

CSV_FILE = "nds_results.csv"

class NDSApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🩺 Patient-Friendly NDS Assessment")
        self.root.geometry("850x750")

        self.scores = {
            "Vibration": 0,
            "Temperature": 0,
            "Pain": 0,
            "Touch": 0,
            "Reflex": 0
        }

        self.test_order = ["Vibration", "Temperature", "Pain", "Touch", "Reflex"]
        self.current_test = 0

        self.frame = tk.Frame(root)
        self.frame.pack(fill="both", expand=True)

        self.instructions_text = (
            "📌 تعليمات عامة قبل البدء:\n"
            "- اجلس في مكان هادئ\n"
            "- رجليك مكشوفة\n"
            "- غمض عينك\n"
            "- ركز أثناء الفحص وأجب بصراحة\n"
            "- كل اختبار لكل قدم، خذ أسوأ نتيجة\n"
            "- استرخي أثناء الفحص"
        )

        self.show_instructions()

    def clear(self):
        for widget in self.frame.winfo_children():
            widget.destroy()

    def show_instructions(self):
        self.clear()
        tk.Label(self.frame, text="🩺 Expanded NDS Home Assessment", font=("Arial", 18, "bold")).pack(pady=10)
        tk.Label(self.frame, text=self.instructions_text, justify="left", font=("Arial", 12)).pack(pady=10)
        tk.Button(self.frame, text="ابدأ الفحص", font=("Arial", 14, "bold"), bg="green", fg="white",
                  command=self.next_test).pack(pady=20)

    def next_test(self):
        if self.current_test < len(self.test_order):
            self.clear()
            test = self.test_order[self.current_test]
            getattr(self, f"{test.lower()}_test")()
            self.current_test += 1
        else:
            self.show_result()

    # ---------------- Timer ----------------
    def start_timer(self, label, duration=10):
        def countdown():
            for i in range(duration, -1, -1):
                label.config(text=f"⏱️ {i} ثانية")
                time.sleep(1)
        threading.Thread(target=countdown, daemon=True).start()

    # ---------------- Add image ----------------
    def add_image(self, path):
        try:
            img = Image.open(path)
            img = img.resize((250, 250))
            photo = ImageTk.PhotoImage(img)
            label = tk.Label(self.frame, image=photo)
            label.image = photo
            label.pack(pady=10)
        except:
            tk.Label(self.frame, text="[الصورة غير موجودة]", font=("Arial", 12, "italic")).pack(pady=10)

    # ---------------- Tests ----------------
    def vibration_test(self):
        instructions = (
            "1️⃣ اختبار الإحساس بالاهتزاز - شرح للمريض:\n\n"
            "1- استخدم الهاتف على وضع الاهتزاز أو ماكينة حلاقة كهربائية.\n"
            "2- اجلس براحة وغمض عينيك.\n"
            "3- ضع على إصبع القدم الكبير أولاً، ثم الكاحل.\n"
            "4- قل فورًا عندما تشعر بالاهتزاز.\n"
            "5- كرر للقدم الأخرى.\n"
            "6- خذ أسوأ نتيجة بين القدمين."
        )
        self.test_ui(
            "Vibration Sensation",
            instructions,
            "images/vibration.png",
            {
                "✅ طبيعي (≤2 ث)": 0,
                "⚠️ ضعيف (3–5 ث)": 2,
                "❌ متأخر (6–10 ث)": 3,
                "❌ غائب (>10 ث)": 6
            },
            "Vibration",
            timer=True
        )

    # ---------------- Temperature Test (4 حالات) ----------------
    def temperature_test(self):
        self.temp_cases = [
            ("يمين", "دافئ"),
            ("يمين", "بارد"),
            ("يسار", "دافئ"),
            ("يسار", "بارد")
        ]
        self.temp_index = 0
        self.temp_values = {}  # لتخزين كل درجة حرارة
        self.show_temp_case()

    def show_temp_case(self):
        self.clear()
        foot, temp_type = self.temp_cases[self.temp_index]
        instructions = (
            f"اختبار الإحساس بالحرارة:\n"
            f"القدم: {foot}\n"
            f"نوع الماء: {temp_type}\n\n"
            "1- اجلس براحة وغمض عينيك.\n"
            "2- استخدم أزرار '+' و'-' لضبط درجة حرارة الماء.\n"
            "3- ضع إصبعك في الماء، وعند شعورك اضغط 'لقد شعرت'.\n"
            "4- بعد الضغط سيتم الانتقال للحالة التالية."
        )

        tk.Label(self.frame, text="Temperature Sensation", font=("Arial", 16, "bold")).pack(pady=10)
        self.add_image("images/temp.png")
        tk.Label(self.frame, text=instructions, justify="left", font=("Arial", 12)).pack(pady=10)

        self.current_temp = 25
        temp_label = tk.Label(self.frame, text=f"درجة حرارة الماء: {self.current_temp}°C", font=("Arial", 14))
        temp_label.pack(pady=5)

        def increase_temp():
            self.current_temp += 1
            temp_label.config(text=f"درجة حرارة الماء: {self.current_temp}°C")

        def decrease_temp():
            self.current_temp -= 1
            temp_label.config(text=f"درجة حرارة الماء: {self.current_temp}°C")

        def record_temp():
            key = f"Temp_{foot}_{temp_type}"
            self.temp_values[key] = self.current_temp
            self.temp_index += 1
            if self.temp_index < len(self.temp_cases):
                self.show_temp_case()
            else:
                self.calculate_temperature_score()

        tk.Button(self.frame, text="+ زيادة درجة الحرارة", command=increase_temp, bg="orange").pack(pady=2)
        tk.Button(self.frame, text="- تقليل درجة الحرارة", command=decrease_temp, bg="lightblue").pack(pady=2)
        tk.Button(self.frame, text="لقد شعرت!", command=record_temp, bg="green", fg="white").pack(pady=10)

    def calculate_temperature_score(self):
        # الأسوأ بين الأربع حالات
        temps = list(self.temp_values.values())
        worst_temp = max([abs(t - 37) for t in temps])  # 37°C طبيعي
        if worst_temp <= 2:
            score = 0
        elif worst_temp <= 5:
            score = 2
        elif worst_temp <= 8:
            score = 3
        else:
            score = 5
        self.scores["Temperature"] = score
        # حفظ جميع القيم الأربع
        self.scores.update(self.temp_values)
        self.next_test()

    # ---------------- باقي الاختبارات ----------------
    def pain_test(self):
        instructions = (
            "3️⃣ اختبار الإحساس بالألم (Pinprick) - شرح للمريض:\n\n"
            "1- اجلس براحة وغمض عينيك.\n"
            "2- استخدم دبوس أو أداة صغيرة للضغط على 5 نقاط مختلفة في القدم.\n"
            "3- اضغط خفيف أولاً، ثم أقوى قليلاً.\n"
            "4- قل 'ألم' إذا شعرت بالألم، أو 'لمس فقط' إذا لم يكن مؤلم.\n"
            "5- كرر للقدم الأخرى.\n"
            "6- خذ أسوأ نتيجة بين القدمين."
        )
        self.test_ui(
            "Pain Sensation",
            instructions,
            "images/pain.png",
            {
                "✅ طبيعي": 0,
                "⚠️ ألم ضعيف": 2,
                "❌ لمس فقط": 4,
                "❌ لا إحساس": 6
            },
            "Pain"
        )

    def touch_test(self):
        instructions = (
            "4️⃣ اختبار اللمس الخفيف - شرح للمريض:\n\n"
            "1- اجلس براحة وغمض عينيك.\n"
            "2- استخدم قطعة قطن أو منديل.\n"
            "3- المس 5 نقاط مختلفة في القدم بلطف.\n"
            "4- قل 'أيوه' عند الإحساس.\n"
            "5- كرر للقدم الأخرى.\n"
            "6- خذ أسوأ نتيجة بين القدمين."
        )
        self.test_ui(
            "Light Touch",
            instructions,
            "images/touch.png",
            {
                "✅ طبيعي": 0,
                "⚠️ ضعيف": 1,
                "❌ متقطع": 2,
                "❌ غائب": 3
            },
            "Touch"
        )

    def reflex_test(self):
        instructions = (
            "5️⃣ اختبار منعكس القدم/التوازن - شرح للمريض:\n\n"
            "1- قف على أطراف أصابعك لمدة 10 ثواني.\n"
            "2- امشي على الكعب ببطء.\n"
            "3- حاول تحافظ على توازنك.\n"
            "4- كرر مع القدم الأخرى.\n"
            "5- قيم قدرتك على التوازن."
        )
        self.test_ui(
            "Functional Reflex",
            instructions,
            "images/reflex.png",
            {
                "✅ طبيعي": 0,
                "⚠️ صعوبة بسيطة": 1,
                "❌ صعوبة واضحة": 2,
                "❌ غير قادر": 3
            },
            "Reflex",
            finish=True
        )

    # ---------------- UI Template ----------------
    def test_ui(self, title, instructions, image_path, options, key, timer=False, finish=False):
        tk.Label(self.frame, text=title, font=("Arial", 16, "bold")).pack(pady=10)
        self.add_image(image_path)
        tk.Label(self.frame, text=instructions, justify="left", font=("Arial", 12)).pack(pady=10)

        if timer:
            timer_label = tk.Label(self.frame, font=("Arial", 14))
            timer_label.pack(pady=10)
            tk.Button(self.frame, text="ابدأ المؤقت 10 ثواني",
                      command=lambda: self.start_timer(timer_label)).pack(pady=5)

        var = tk.IntVar()
        for text, val in options.items():
            tk.Radiobutton(self.frame, text=text, variable=var, value=val, anchor="w", justify="left").pack(fill="x")

        btn_text = "انهاء" if finish else "التالي"
        tk.Button(self.frame, text=btn_text,
                  command=lambda: self.save_and_next(key, var.get()), font=("Arial", 12), bg="blue", fg="white").pack(pady=15)

    # ---------------- Save Score ----------------
    def save_and_next(self, key, value):
        self.scores[key] = value
        self.next_test()

    # ---------------- Result + Save CSV ----------------
    def show_result(self):
        self.clear()
        total = sum(self.scores.get(k, 0) for k in ["Vibration", "Temperature", "Pain", "Touch", "Reflex"])

        if total <= 5:
            severity = "طبيعي"
        elif total <= 10:
            severity = "خفيف"
        elif total <= 16:
            severity = "متوسط"
        else:
            severity = "شديد"

        tk.Label(self.frame, text="✅ الفحص اكتمل", font=("Arial", 18, "bold")).pack(pady=20)
        tk.Label(self.frame, text=f"NDS Score: {total} / 23", font=("Arial", 16)).pack()
        tk.Label(self.frame, text=f"Severity: {severity}", font=("Arial", 16, "bold")).pack(pady=10)

        self.save_to_csv(total, severity)
        messagebox.showinfo("Saved", "✅ تم حفظ النتائج بنجاح في nds_results.csv")

    def save_to_csv(self, total, severity):
        record = self.scores.copy()
        record["NDS_total"] = total
        record["Severity"] = severity
        record["Date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        df_new = pd.DataFrame([record])

        if os.path.exists(CSV_FILE):
            df_old = pd.read_csv(CSV_FILE)
            df = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df = df_new

        df.to_csv(CSV_FILE, index=False)


# Run App
root = tk.Tk()
app = NDSApp(root)
root.mainloop()


In [15]:
import pandas as pd
import os
from datetime import datetime

CSV_FILE = "nds_results.csv"

class NDSAssessment:
    def __init__(self):
        self.scores = {
            "Vibration": 0,
            "Temperature": 0,
            "Pain": 0,
            "Touch": 0,
            "Reflex": 0
        }
        self.temp_values = {}

    # ---------------- Temperature Test ----------------
    def temperature_test(self):
        temp_cases = [
            ("يمين", "دافئ"),
            ("يمين", "بارد"),
            ("يسار", "دافئ"),
            ("يسار", "بارد")
        ]
        print("\n2️⃣ اختبار الإحساس بالحرارة والبرودة\n")
        for foot, temp_type in temp_cases:
            while True:
                try:
                    temp = float(input(f"القدم: {foot} | نوع الماء: {temp_type} | أدخل درجة الحرارة التي شعرت بها: "))
                    break
                except ValueError:
                    print("⚠️ ادخل قيمة صحيحة.")
            key = f"Temp_{foot}_{temp_type}"
            self.temp_values[key] = temp

        # حساب الأسوأ
        temps = list(self.temp_values.values())
        worst_temp = max([abs(t - 37) for t in temps])  # 37°C طبيعي
        if worst_temp <= 2:
            score = 0
        elif worst_temp <= 5:
            score = 2
        elif worst_temp <= 8:
            score = 3
        else:
            score = 5

        self.scores["Temperature"] = score
        self.scores.update(self.temp_values)
        print(f"✅ درجة NDS للحرارة بعد الأربع حالات: {score}")

    # ---------------- بقية الاختبارات ----------------
    def input_test(self, name, options):
        print(f"\nاختبار {name}")
        for i, (desc, val) in enumerate(options.items(), 1):
            print(f"{i}. {desc} (نقاط: {val})")
        while True:
            try:
                choice = int(input("اختر رقم الخيار المناسب: "))
                if 1 <= choice <= len(options):
                    self.scores[name] = list(options.values())[choice-1]
                    break
                else:
                    print("⚠️ اختر رقم صحيح.")
            except ValueError:
                print("⚠️ ادخل رقم صحيح.")

    def vibration_test(self):
        options = {
            "✅ طبيعي (≤2 ث)": 0,
            "⚠️ ضعيف (3–5 ث)": 2,
            "❌ متأخر (6–10 ث)": 3,
            "❌ غائب (>10 ث)": 6
        }
        self.input_test("Vibration", options)

    def pain_test(self):
        options = {
            "✅ طبيعي": 0,
            "⚠️ ألم ضعيف": 2,
            "❌ لمس فقط": 4,
            "❌ لا إحساس": 6
        }
        self.input_test("Pain", options)

    def touch_test(self):
        options = {
            "✅ طبيعي": 0,
            "⚠️ ضعيف": 1,
            "❌ متقطع": 2,
            "❌ غائب": 3
        }
        self.input_test("Touch", options)

    def reflex_test(self):
        options = {
            "✅ طبيعي": 0,
            "⚠️ صعوبة بسيطة": 1,
            "❌ صعوبة واضحة": 2,
            "❌ غير قادر": 3
        }
        self.input_test("Reflex", options)

    # ---------------- حساب النتيجة ----------------
    def calculate_result(self):
        total = sum([self.scores[k] for k in ["Vibration", "Temperature", "Pain", "Touch", "Reflex"]])
        if total <= 5:
            severity = "طبيعي"
        elif total <= 10:
            severity = "خفيف"
        elif total <= 16:
            severity = "متوسط"
        else:
            severity = "شديد"

        print("\n✅ الفحص اكتمل")
        print(f"NDS Score: {total} / 23")
        print(f"Severity: {severity}")
        return total, severity

    # ---------------- حفظ CSV ----------------
    def save_to_csv(self, total, severity):
        record = self.scores.copy()
        record["NDS_total"] = total
        record["Severity"] = severity
        record["Date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        df_new = pd.DataFrame([record])

        if os.path.exists(CSV_FILE):
            df_old = pd.read_csv(CSV_FILE)
            df = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df = df_new

        df.to_csv(CSV_FILE, index=False)
        print(f"✅ تم حفظ النتائج بنجاح في {CSV_FILE}")


# ---------------- Run Assessment ----------------
assessment = NDSAssessment()
assessment.vibration_test()
assessment.temperature_test()
assessment.pain_test()
assessment.touch_test()
assessment.reflex_test()

total, severity = assessment.calculate_result()
assessment.save_to_csv(total, severity)



اختبار Vibration
1. ✅ طبيعي (≤2 ث) (نقاط: 0)
2. ⚠️ ضعيف (3–5 ث) (نقاط: 2)
3. ❌ متأخر (6–10 ث) (نقاط: 3)
4. ❌ غائب (>10 ث) (نقاط: 6)


اختر رقم الخيار المناسب:  3



2️⃣ اختبار الإحساس بالحرارة والبرودة



القدم: يمين | نوع الماء: دافئ | أدخل درجة الحرارة التي شعرت بها:  42
القدم: يمين | نوع الماء: بارد | أدخل درجة الحرارة التي شعرت بها:  20
القدم: يسار | نوع الماء: دافئ | أدخل درجة الحرارة التي شعرت بها:  41
القدم: يسار | نوع الماء: بارد | أدخل درجة الحرارة التي شعرت بها:  20


✅ درجة NDS للحرارة بعد الأربع حالات: 5

اختبار Pain
1. ✅ طبيعي (نقاط: 0)
2. ⚠️ ألم ضعيف (نقاط: 2)
3. ❌ لمس فقط (نقاط: 4)
4. ❌ لا إحساس (نقاط: 6)


...


اختبار Touch
1. ✅ طبيعي (نقاط: 0)
2. ⚠️ ضعيف (نقاط: 1)
3. ❌ متقطع (نقاط: 2)
4. ❌ غائب (نقاط: 3)


اختر رقم الخيار المناسب:  2



اختبار Reflex
1. ✅ طبيعي (نقاط: 0)
2. ⚠️ صعوبة بسيطة (نقاط: 1)
3. ❌ صعوبة واضحة (نقاط: 2)
4. ❌ غير قادر (نقاط: 3)


اختر رقم الخيار المناسب:  1



✅ الفحص اكتمل
NDS Score: 13 / 23
Severity: متوسط
✅ تم حفظ النتائج بنجاح في nds_results.csv
